# 🚀 02. Train Vision-Only Student Model (Phase 2 - CMKD)

> **Project:** Durian Ripeness Classification via Cross-Modal Knowledge Distillation (CMKD)  
> **Lead:** Leader / Member 1 (Vision Lead & Multi-Modal Framework)  
> **Reference:** `specification/project_specification.pdf` (Section 1.3.3) & `tmp_student.md`  
> **Notebook Role:**
1. Mount Google Drive & verify GPU availability.
2. `git clone` / `git pull` the latest source code from GitHub.
3. Link dataset (`data/final`), Teacher checkpoint (`checkpoints/teachermodal/best_teacher.pth`), and create target folders (`checkpoints/studentmodal`, `results_student`) on Google Drive.
4. Launch 2-phase distillation training (`train_student.py`) and export the benchmark comparison table.

In [1]:
# ==============================================================================
# 1. MOUNT GOOGLE DRIVE & CHECK GPU
# ==============================================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("✅ Google Drive mounted successfully!")
else:
    print("ℹ️ Running in Local environment.")

!nvidia-smi

Mounted at /content/drive
✅ Google Drive mounted successfully!
Fri Nov 20 19:37:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |         

In [ ]:
# ==============================================================================
# 2. SYNC LATEST CODE FROM GITHUB
# ==============================================================================
REPO_URL = "https://github.com/Cocobaut/durian-ripeness-classification.git"
BRANCH = "cocobaut"
REPO_DIR = "/content/durian-ripeness-classification"

if IN_COLAB:
    %cd /content
    if not os.path.exists(REPO_DIR):
        if BRANCH:
            !git clone -b $BRANCH $REPO_URL $REPO_DIR
        else:
            !git clone $REPO_URL $REPO_DIR
        %cd $REPO_DIR
    else:
        %cd $REPO_DIR
        # Remove temporary repo symlinks (without touching Drive files) to prevent git pull conflicts
        !find . -maxdepth 5 -type l -delete
        !git fetch origin
        if BRANCH:
            !git checkout -f $BRANCH
            !git reset --hard origin/$BRANCH
        else:
            !git reset --hard origin/$(git rev-parse --abbrev-ref HEAD)
else:
    cur = os.path.abspath(".")
    while not (os.path.isdir(os.path.join(cur, "src")) and os.path.isdir(os.path.join(cur, "workspaces"))):
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    os.chdir(cur)

print(f"✅ Working in project root directory: {os.getcwd()}")

/content
Cloning into '/content/durian-ripeness-classification'...
remote: Enumerating objects: 818, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (143/143), done.
remote: Total 818 (delta 36), reused 139 (delta 32), pack-reused 632 (from 1)
Receiving objects: 100% (818/818), 69.83 MiB | 20.64 MiB/s, done.
Resolving deltas: 100% (305/305), done.
/content/durian-ripeness-classification
✅ Working in project root directory: /content/durian-ripeness-classification


In [3]:
# ==============================================================================
# 3. FAST DEPENDENCY INSTALLATION WITH `uv`
# ==============================================================================
!pip install -q uv
!uv pip install --system -r requirements.txt
print("✅ Installed all dependencies successfully!")

Using Python 3.13.16 environment at: /usr
Resolved 184 packages in 771ms                                       
Prepared 15 packages in 419ms                                            
Uninstalled 3 packages in 22ms
Installed 15 packages in 134ms                              
 + async-lru==2.4.0
 + comm==0.2.3
 - ipykernel==6.17.1
 + ipykernel==7.4.0
 + jedi==0.20.1
 + json5==0.17.3
 + jupyter==1.1.1
 + jupyter-builder==1.3.0
 - jupyter-client==7.4.9
 + jupyter-client==8.10.0
 + jupyter-lsp==2.3.1
 + jupyterlab==4.6.6
 + jupyterlab-server==2.28.1
 + nest-asyncio2==1.7.4
 - notebook==6.5.7
 + notebook==7.6.5
 + pydeck==0.9.3
 + streamlit==1.67.0
✅ Installed all dependencies successfully!


In [4]:
# ==============================================================================
# 4. CONFIGURE CHECKPOINT & DATASET LINKS FROM GOOGLE DRIVE
# ==============================================================================
from pathlib import Path

if IN_COLAB:
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/durian-ripeness-cmkd")
    DRIVE_CHECKPOINTS = DRIVE_PROJECT_ROOT / "checkpoints"
    DRIVE_DATA = DRIVE_PROJECT_ROOT / "data" / "final"
    DRIVE_RESULTS_TEACHER = DRIVE_PROJECT_ROOT / "results_teacher"
    DRIVE_RESULTS_STUDENT = DRIVE_PROJECT_ROOT / "results_student"

    # Pre-create studentmodal & results_student folders on Google Drive for real-time persistence
    (DRIVE_CHECKPOINTS / "teachermodal").mkdir(parents=True, exist_ok=True)
    (DRIVE_CHECKPOINTS / "studentmodal").mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTS_STUDENT.mkdir(parents=True, exist_ok=True)

    # Symlink repo directories to Google Drive so outputs persist across Colab disconnects
    !rm -rf checkpoints/unimodal checkpoints/teachermodal checkpoints/studentmodal
    !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    !mkdir -p checkpoints data/final
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'unimodal')}" checkpoints/unimodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'teachermodal')}" checkpoints/teachermodal
    !ln -sfn "{str(DRIVE_CHECKPOINTS / 'studentmodal')}" checkpoints/studentmodal
    !ln -sfn "{str(DRIVE_RESULTS_STUDENT)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results_student
    if DRIVE_RESULTS_TEACHER.exists():
        !rm -rf workspaces/member1_vision_lead/04_train_teacher_and_student/results
        !ln -sfn "{str(DRIVE_RESULTS_TEACHER)}" workspaces/member1_vision_lead/04_train_teacher_and_student/results
    for split_name in ["train", "valid", "test"]:
        src_split = DRIVE_DATA / split_name
        if src_split.exists():
            !rm -rf data/final/{split_name}
            !ln -sfn "{str(src_split)}" data/final/{split_name}

Path("checkpoints/studentmodal").mkdir(parents=True, exist_ok=True)
print("✅ Verifying configured paths:")
print(" - Teacher ckpt   :", Path("checkpoints/teachermodal/best_teacher.pth").exists(), "->", Path("checkpoints/teachermodal").resolve())
print(" - Studentmodal   :", Path("checkpoints/studentmodal").resolve())
print(" - Results student:", Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student").resolve())
print(" - Manifest CSV   :", Path("data/final/final_multimodal_manifest.csv").exists())

✅ Verifying configured paths:
 - Teacher ckpt   : True -> /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/teachermodal
 - Studentmodal   : /content/drive/MyDrive/durian-ripeness-cmkd/checkpoints/studentmodal
 - Results student: /content/drive/MyDrive/durian-ripeness-cmkd/results_student
 - Manifest CSV   : True


In [5]:
# ==============================================================================
# 5. RUN MULTI-VIEW VISION-ONLY STUDENT DISTILLATION TRAINING (PHASE 2 - CMKD)
# ==============================================================================
!python workspaces/member1_vision_lead/04_train_teacher_and_student/train_student.py \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --teacher-ckpt checkpoints/teachermodal/best_teacher.pth \
    --student-backbone convnextv2_base \
    --save-path checkpoints/studentmodal/best_student.pth \
    --results-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results_student \
    --alpha 1.0 \
    --beta 2.0 \
    --gamma 1.0 \
    --temperature 3.0 \
    --feat-loss-type cosine \
    --loss-type cost_sensitive_focal \
    --focal-gamma 2.0 \
    --cost-weight 0.5 \
    --batch-size 4 \
    --accum-steps 2 \
    --epochs 30 \
    --warmup-epochs 3 \
    --lr 5e-4 \
    --lr-backbone 1.5e-5 \
    --min-lr 1e-6 \
    --patience 10 \
    --select-metric fruit_macro_f1 \
    --eval-test \
    --device cuda

[2026-11-20 19:38:05] ==============================================================================
[2026-11-20 19:38:05] TRAINING VISION-ONLY STUDENT VIA CMKD (STAGE 2)
[2026-11-20 19:38:05] Device: cuda | AMP: True (torch.bfloat16) | batch 4 x accum 2
[2026-11-20 19:38:05] Student backbone : convnextv2_base.fcmae_ft_in1k
[2026-11-20 19:38:05] Distill weights  : alpha=1.0 (CE) | beta=2.0 (KD, T=3.0) | gamma=1.0 (Feat: cosine)
[2026-11-20 19:38:05] Best student ckpt: /content/durian-ripeness-classification/checkpoints/studentmodal/best_student.pth
[2026-11-20 19:38:05] ==============================================================================
[2026-11-20 19:38:05] Loading Frozen Teacher from: /content/durian-ripeness-classification/checkpoints/teachermodal/best_teacher.pth
Traceback (most recent call last):
  File "/content/durian-ripeness-classification/workspaces/member1_vision_lead/04_train_teacher_and_student/train_student.py", line 696, in <module>
    main()
    ~~~~^^
  Fil

In [6]:
# ==============================================================================
# 6. INDEPENDENT RGB-ONLY EVALUATION & DISPLAY BENCHMARK COMPARISON TABLE
# ==============================================================================
!python workspaces/member1_vision_lead/04_train_teacher_and_student/eval_student.py \
    --checkpoint checkpoints/studentmodal/best_student.pth \
    --manifest data/final/final_multimodal_manifest.csv \
    --data-dir data/final \
    --split test \
    --output-dir workspaces/member1_vision_lead/04_train_teacher_and_student/results_student

comp_md = Path("workspaces/member1_vision_lead/04_train_teacher_and_student/results_student/benchmark_comparison.md")
if comp_md.is_file():
    print("\n" + comp_md.read_text(encoding="utf-8"))
print("✅ All Student evaluation results saved at:", comp_md.parent.resolve())

Traceback (most recent call last):
  File "/content/durian-ripeness-classification/workspaces/member1_vision_lead/04_train_teacher_and_student/eval_student.py", line 291, in <module>
    main()
    ~~~~^^
  File "/content/durian-ripeness-classification/workspaces/member1_vision_lead/04_train_teacher_and_student/eval_student.py", line 267, in main
    res = evaluate_student_split(
        ckpt_path=args.checkpoint,
    ...<9 lines>...
        seed=args.seed,
    )
  File "/content/durian-ripeness-classification/workspaces/member1_vision_lead/04_train_teacher_and_student/eval_student.py", line 119, in evaluate_student_split
    model = build_student_from_checkpoint(ckpt_path, device=device_obj, eval_mode=True)
  File "/content/durian-ripeness-classification/src/models/student_model.py", line 225, in build_student_from_checkpoint
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
  File "/usr/local/lib/python3.13/dist-packages/torch/serialization.py", line 1565, in l